In [ ]:
# import libraries
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
from scipy.stats import chi2
import healpy as hp

data_path = "/data/test_newrepo" 

In [ ]:
# activate this cell to create the LUTs, then disable it.

#Example
file_name = "run72_hard_dc3_shared"
lut_name = "hard_lut"

file_path = data_path+'/'+file_name+'_dataset.pkl'
# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array = pickle.load(file)
        
print(file_path)
print(len(loaded_array))

m_tables = np.empty((len(loaded_array),8))

count = -1
for element in loaded_array:
    count+=1
    m_tables[count] = element['counts']+element['coord']

m_tables = m_tables[:count+1]

np.save(data_path+'/LUT_'+lut_name+'.npy', m_tables)                
print(data_path+'/LUT/LUT_'+lut_name+'.npy')

In [ ]:
#Load LUTs

lut_name_soft = "soft_lut"
lut_name_medium = "medium_lut"
lut_name_hard = "hard_lut"

lut_soft = np.load(data_path+'/LUT_'+lut_name_soft+'.npy')
lut_medium = np.load(data_path+'/LUT_'+lut_name_medium+'.npy')
lut_hard = np.load(data_path+'/LUT_'+lut_name_hard+'.npy')

In [ ]:
lut_medium.shape

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
# Plot della mappa
hp.mollview(lut_medium[:,3], title="Medium Y1", unit="counts", norm='hist',nest=True )
plt.show()

In [ ]:
lut_plot = lut_medium[:,4]

hp.projview(
    lut_plot,
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    badcolor="antiquewhite",
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14  # qui imposti il font size dei numeri della colorbar
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 0
    }
    
)

ax = plt.gca()

fig = plt.gcf()
axes = fig.get_axes()
main_ax = plt.gca()

for ax in axes:
    if ax != main_ax:
        cbar_ax = ax
        break

cbar_ax.set_xlabel("LUT example", fontsize=14) 

plt.show()

In [ ]:
#test with LUTs
test_dataset_name = "run64" #change this according to the dataset you want to test
file_path = data_path+'/'+test_dataset_name+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)
    
test_dataset = loaded_array_test

In [ ]:
# Leave the filters to 0 if you want to test all the dataset otherwise set the filters to 1 and change the parameters

filter_flux = 0
filter_spectra = 0

filter_theta=0
filter_phi=0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] >14 and grb['flux'] <= 16:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        # filter by spectrum using the spectrum parameter
        if  "230" in grb['spectrum']: 
        #["Band 10 10000 -1.9 -3.7 230","Band 10 10000 -1 -2.3 699.9","Comptonized 10 10000 -0.5 1500"]
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_theta==1:
    print("filter theta")
    for grb in loaded_array_test:
        if float(grb['coord'][0])>50 and float(grb['coord'][0])<130:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_phi==1:
    print("filter phi")
    for grb in loaded_array_test:
        if float(grb['coord'][1])>100 and float(grb['coord'][1])<170:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
def get_radians(coords):
    # Unpack the list of (theta, phi) pairs
    theta, phi = zip(*coords)
    
    # Convert to numpy arrays
    theta = np.array(theta)
    phi = np.array(phi)
    
    # Wrap phi values >180 into the range (-180, 180]
    mask = phi > 180
    phi[mask] -= 360
    
    # Convert theta into colatitude (90 - theta)
    theta = 90 - theta

    # Return values in radians
    return np.radians(theta), np.radians(phi)

def calculate_chi_squared_optimized(s, b, m):
    """
    Compute the chi-squared value for each position in the grid in an optimized way.

    Parameters:
        s: array of shape (6,) with observed counts (s(j)).
        b: array of shape (12,) with background counts (b(j)).
        m: array of shape (12, 41168) with model counts (m(j, i)).

    Returns:
        chi_squared: array of shape (41168,) with the chi-squared value for each position i.
    """
    # Use only the first 6 detectors
    indices = np.arange(6)
    
    # Extract model counts for these detectors
    m_subset = m[:, indices]  # Shape: (41168, 6)

    # Compute numerator and denominator for normalization factor f_i
    with np.errstate(divide='ignore', invalid='ignore'):
        numerator = np.sum(m_subset * (s[indices] - b[indices]) / s[indices], axis=1)
        denominator = np.sum((m_subset**2) / s[indices], axis=1)

        # Avoid division by zero
        f_i = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator != 0)

        # Expand f_i for broadcasting
        f_i_expanded = f_i[:, np.newaxis]  # Shape: (41168, 1)

        # Compute chi-squared terms
        chi_numerator = (s[indices] - b[indices] - f_i_expanded * m_subset)**2
        chi_denominator = b[indices] + f_i_expanded * m_subset

        # Safe division for chi-squared elements
        chi_squared_elements = np.divide(
            chi_numerator, chi_denominator,
            out=np.full_like(chi_numerator, np.finfo(np.float64).max),
            where=chi_denominator != 0
        )

    # Sum over detectors to obtain chi^2 per direction
    chi_squared = np.sum(chi_squared_elements, axis=1)

    return chi_squared


def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

In [ ]:
def find_closest_pixel(theta_real, phi_real, best_lut):
    # Convert input angles from degrees to radians
    theta_real_rad = np.deg2rad(theta_real)
    phi_real_rad = np.deg2rad(phi_real)
    
    # Extract θ and φ from best_lut (last two columns, in degrees) and convert to radians
    theta_vals_m = np.deg2rad(best_lut[:, -2])
    phi_vals_m = np.deg2rad(best_lut[:, -1])
    
    # Compute the cosine of the spherical angular distance
    cos_dist = (
        np.sin(theta_real_rad) * np.sin(theta_vals_m) * np.cos(phi_real_rad - phi_vals_m)
        + np.cos(theta_real_rad) * np.cos(theta_vals_m)
    )

    # Compute the angular distance (in radians)
    angular_distance = np.arccos(cos_dist)
    
    # Find the index of the minimum distance
    min_idx = np.argmin(angular_distance)
    
    # The index of the maximum cosine corresponds to the minimum distance
    # min_idx = np.argmax(cos_dist)
    return min_idx

In [ ]:
def analyze_grbs(test_dataset, with_background):
    """
    Analyze a dataset of GRBs:
      - pick the best spectrum (soft/medium/hard) via chi-squared minimization,
      - localize using the argmin index on the best LUT,
      - compute angular/coordinate errors,
      - collect stats on whether the chosen spectrum matches the true one.

    Returns:
        results: list of [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist, global_min, real_chi2, chi2_array]
        spectra_fitted: int, number of correctly identified spectra
        good_spectra_fit_distances: list of angular distances for correct spectrum choices
        bad_spectra_fit_distances: list of angular distances for incorrect spectrum choices
    """
    results = []
    spectra_fitted = 0
    good_spectra_fit_distances = []
    bad_spectra_fit_distances = []

    # Map spectrum string to label
    def classify_spectrum(spectrum_str: str) -> str:
        if "230" in spectrum_str:
            return "soft"
        if "699.9" in spectrum_str:
            return "medium"
        if "Compton" in spectrum_str:
            return "hard"
        return "random"

    # LUTs registry (expects lut_soft, lut_medium, lut_hard to be defined)
    lut_by_key = {
        "soft": lut_soft,
        "medium": lut_medium,
        "hard": lut_hard,
    }
    keys = ["soft", "medium", "hard"]

    count = 0
    for grb in test_dataset:
        count += 1
        if count % 1000 == 0:
            print(count)

        counts = np.array(grb["counts"], dtype=float)
        spectrum_key_true = classify_spectrum(grb["spectrum"])

        theta_real = float(grb["coord"][0])
        phi_real = float(grb["coord"][1])

        # Compute chi^2 arrays for each spectrum key
        chi2_by_key = {}
        mins = []
        if with_background:
            # Fixed background vector as in your code
            b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617], dtype=float)
            b_vec = b_sim * 20.0

            for k in keys:
                # Draw independent Poisson noise for each spectrum (same as original behavior)
                noisy_counts = counts + np.random.poisson(b_vec)
                chi2 = calculate_chi_squared_optimized(noisy_counts, b_vec, lut_by_key[k])
                chi2_by_key[k] = chi2
                mins.append(np.min(chi2))
        else:
            zero_b = np.zeros(6, dtype=float)
            for k in keys:
                chi2 = calculate_chi_squared_optimized(counts, zero_b, lut_by_key[k])
                chi2_by_key[k] = chi2
                mins.append(np.min(chi2))

        # Pick global best spectrum via argmin
        best_idx = int(np.argmin(mins))
        best_key = keys[best_idx]
        chi2_array = chi2_by_key[best_key]
        best_lut = lut_by_key[best_key]

        # Check if predicted spectrum matches the labeled one
        spectra_selected = (spectrum_key_true == best_key)
        if spectra_selected and best_key in {"soft", "medium", "hard"}:
            spectra_fitted += 1

        # Best location from LUT argmin
        argmin_index = int(np.argmin(chi2_array))
        theta_loc = best_lut[argmin_index][6]
        phi_loc = best_lut[argmin_index][7]

        # Chi^2 at the true direction (closest pixel to the real coordinates)
        min_idx = find_closest_pixel(theta_real, phi_real, best_lut)
        real_chi2 = chi2_array[min_idx]

        # Angular and component-wise distances
        dist = angular_distance(theta_loc, phi_loc, theta_real, phi_real)
        theta_dist = np.abs(theta_loc - theta_real)
        phi_dist = diff_phi(phi_loc, phi_real)

        if spectra_selected:
            good_spectra_fit_distances.append(dist)
        else:
            bad_spectra_fit_distances.append(dist)

        results.append([
            theta_real, phi_real,
            theta_loc, phi_loc,
            dist, theta_dist, phi_dist,
            mins[best_idx], real_chi2, chi2_array
        ])

    return results, spectra_fitted, good_spectra_fit_distances, bad_spectra_fit_distances

In [ ]:
# With True we have background
results,spectra_fitted, good_spectra_fit_distances,bad_spectra_fit_distances= analyze_grbs(test_dataset,False)

In [ ]:
# Collect angular errors and 90% containment areas
distances = []
theta_distances = []
phi_distances = []
areas_90 = []

# Constant Δchi2 for 90% CL with 2 dof
delta_chi2_90 = chi2.ppf(0.9, 2)

for res in results:
    # Unpack metrics
    dist = res[4]
    dtheta = res[5]
    dphi = res[6]
    chi2_min = res[7]
    chi2_map = res[9]  # array-like chi² map over sky pixels (Healpix)

    distances.append(dist)
    theta_distances.append(dtheta)
    phi_distances.append(dphi)

    # Threshold for the 90% region: chi2 < chi2_min + Δchi2
    limit = chi2_min + delta_chi2_90

    # Healpix bookkeeping
    nside = hp.get_nside(chi2_map)
    pix_area_deg2 = hp.nside2pixarea(nside, degrees=True)

    # Pixels inside the 90% region
    mask_inside = chi2_map < limit
    npix_in_region = np.sum(mask_inside)

    # 90% containment area (deg²)
    area_90 = npix_in_region * pix_area_deg2
    areas_90.append(area_90)

# Summary statistics
print(np.mean(areas_90))
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))


In [ ]:
print(np.mean(good_spectra_fit_distances))
print(np.mean(bad_spectra_fit_distances))
print(spectra_fitted/len(test_dataset)*100)

In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
import pickle
hp.projview(
    np.array(areas_90),
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14 
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 5
    }
    
)
plt.show()

In [ ]:
test_labels = []
for grb in test_dataset:
    theta_real = float(grb['coord'][0])
    phi_real = float(grb['coord'][1])
    test_labels.append([theta_real,phi_real])
test_labels = np.array(test_labels)

In [ ]:
theta_rad_test,phi_rad_test = get_radians(test_labels)

In [ ]:
def plot_aitoff(phi_rad,theta_rad,dist,title):

    plt.figure(figsize=(15,10))
    plt.subplot(111, projection="aitoff")
    plt.title("")
    plt.grid(True)
    
    scatter = plt.scatter(phi_rad, theta_rad, c=dist, cmap='gnuplot2', alpha=0.6)
    # Adding color legend
    cbar = plt.colorbar(scatter, orientation='vertical', aspect=20, shrink=0.5)
    cbar.set_label('Error (°)')
    
    plt.title(title)
    plt.subplots_adjust(top=0.95,bottom=0.0)
    plt.show()

In [ ]:
# After reducing the number of GRBs in the dataset it is possible to plot the results

#plot_aitoff(phi_rad_test,theta_rad_test,areas_90,"Area 90% c.l. results")
#plot_aitoff(phi_rad_test,theta_rad_test,distances,"Error results")
#plot_aitoff(phi_rad_test,theta_rad_test,theta_distances,"Theta errors")
#plot_aitoff(phi_rad_test,theta_rad_test,phi_distances,"Phi errors")


In [ ]:
# Choose the index to plot. Eventually it is possible to implement a for cycle.
index = 11367
min_chi2 = results[index][7]
theta_real = float(results[index][0])
phi_real = float(results[index][1])
theta_reco=float(results[index][2])
phi_reco=float(results[index][3])

if phi_real>180:
    phi_real = phi_real-360
if phi_reco>180:
    phi_reco = phi_reco-360

m = results[index][9]

limit = min_chi2 + chi2.ppf(0.9, 2)

m_masked = np.ma.masked_where(m  >= limit , m)

hp.projview(
    m_masked,
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="viridis",
    nest=True,
    unit="", 
    badcolor="antiquewhite",
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14  # qui imposti il font size dei numeri della colorbar
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 0
    }
    
)

hp.newprojplot(theta=np.radians(theta_real), phi=np.radians(phi_real), marker="*", color="magenta", markersize=13)

hp.newprojplot(theta=np.radians(theta_reco), phi=np.radians(phi_reco), marker="X", color="r", markersize=12)

ax = plt.gca()

fig = plt.gcf()
axes = fig.get_axes()
main_ax = plt.gca()

for ax in axes:
    if ax != main_ax:
        cbar_ax = ax
        break

start = int(np.floor(m_masked.min()))
end = int(np.ceil(m_masked.max()))

cbar_ax.set_xlabel("$\chi^2$ value (90% c.l.)", fontsize=14) 

plt.show()

In [ ]:
#Export the data to be plotted with plots/plot_aitoff.py
if  False:
    index = 16257
    # Salva l'array in un file usando pickle
    with open(data_path+"/areas_chi2_"+file_name_test+"_"+str(index)+"_"+str(results[index][0])+"_"+str(results[index][1])+"_"+str(results[index][2])+"_"+str(results[index][3])+"_"+str(round(results[index][7],4))+".pkl", "wb") as f:
        pickle.dump(results[index][9], f)


In [ ]:
# Define 5-degree intervals
intervals = np.arange(0, 185, 5)

# Group counts based on 5-degree intervals
grouped_counts = np.zeros((len(intervals)))
counts = np.zeros((len(intervals)))

total_count = 0
for theta, phi, area in zip(test_labels[:, 0], test_labels[:, 1], areas_90):
    
    if True:  # (phi > 125 and phi < 145) or 
        total_count += 1
        interval_index = int(theta // 5)
        grouped_counts[interval_index] = grouped_counts[interval_index] + area
        counts[interval_index] += 1

# Compute the mean of counts in each interval
grouped_counts = grouped_counts[:total_count]
counts = counts[:total_count]

# Plot the histogram
plt.figure(figsize=(10, 6))
plt.bar(
    intervals[:-1],
    grouped_counts[:-1] / counts[:-1],
    width=5,
    align='edge',
    alpha=1,
    label="90% c.l. error area"
)
plt.xlabel('Theta (deg)', fontsize=15)
plt.ylabel('90% c.l. error area (deg$^2$)', fontsize=15)
plt.title('90% c.l. error area as a function of Theta', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
# plt.legend(fontsize=13)
plt.grid(True)
# plt.legend()
plt.show()


In [ ]:
#Export data to create the histogram
if(False):

    training_histo = {'test_labels': test_labels,"distances":distances,"areas_90":areas_90}

    with open(data_path+"/histo_chi2_area_nobkg.pkl", "wb") as f:
        pickle.dump(training_histo, f)

In [ ]:
# Define 5-degree intervals
intervals = np.arange(0, 185, 5)

# Group values based on 5-degree intervals
grouped_values = np.zeros((len(intervals)))
counts = np.zeros((len(intervals)))

total_count = 0
for theta, phi, value in zip(test_labels[:, 0], test_labels[:, 1], distances):
    
    if True:  # (phi > 125 and phi < 145) or 
        total_count += 1
        interval_index = int(theta // 5)
        grouped_values[interval_index] = grouped_values[interval_index] + value
        counts[interval_index] += 1

# Compute the mean of the values in each interval
grouped_values = grouped_values[:total_count]
counts = counts[:total_count]

# Plot the histogram
plt.figure(figsize=(10, 6))
plt.bar(
    intervals[:-1],
    grouped_values[:-1] / counts[:-1],
    width=5,
    align='edge',
    alpha=1,
    label="loc. error"
)
plt.xlabel('Theta (°)', fontsize=15)
plt.ylabel('Loc. error (°)', fontsize=15)
plt.title('Loc. error as a function of theta', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.grid(True)
# plt.legend()
plt.show()


In [ ]:
step = 10

# Define intervals of 10 degrees
intervals = np.arange(0, 365, step)

# Group counts based on 10-degree intervals
grouped_counts_1 = np.zeros((len(intervals)))
counts_1 = np.zeros((len(intervals)))

grouped_counts_2 = np.zeros((len(intervals)))
counts_2 = np.zeros((len(intervals)))

grouped_counts_3 = np.zeros((len(intervals)))
counts_3 = np.zeros((len(intervals)))

grouped_counts_4 = np.zeros((len(intervals)))
counts_4 = np.zeros((len(intervals)))

total_count_1 = 0
total_count_2 = 0
total_count_3 = 0
total_count_4 = 0

for theta, phi, value in zip(test_labels[:, 0], test_labels[:, 1], areas_90):
    
    if (theta > 20 and theta < 55):
        total_count_1 += 1
        interval_index = int(phi // step)
        grouped_counts_1[interval_index] = grouped_counts_1[interval_index] + value
        counts_1[interval_index] += 1

    if (theta > 55 and theta < 130):
        total_count_2 += 1
        interval_index = int(phi // step)
        grouped_counts_2[interval_index] = grouped_counts_2[interval_index] + value
        counts_2[interval_index] += 1

    if (theta > 130 and theta < 180):
        total_count_3 += 1
        interval_index = int(phi // step)
        grouped_counts_3[interval_index] = grouped_counts_3[interval_index] + value
        counts_3[interval_index] += 1

# Compute the mean of counts in each interval
grouped_counts_1 = grouped_counts_1[:total_count_1]
counts_1 = counts_1[:total_count_1]

grouped_counts_2 = grouped_counts_2[:total_count_2]
counts_2 = counts_2[:total_count_2]

grouped_counts_3 = grouped_counts_3[:total_count_3]
counts_3 = counts_3[:total_count_3]

# Plot the histogram
plt.figure(figsize=(10, 6))
plt.bar(
    intervals[:-1],
    grouped_counts_1[:-1] / counts_1[:-1],
    width=step,
    align='edge',
    alpha=0.5,
    label="loc. error theta=[20°,55°]"
)

plt.bar(
    intervals[:-1],
    grouped_counts_2[:-1] / counts_2[:-1],
    width=step,
    align='edge',
    alpha=0.5,
    label="loc. error theta=[55°,150°]"
)

plt.bar(
    intervals[:-1],
    grouped_counts_3[:-1] / counts_3[:-1],
    width=step,
    align='edge',
    alpha=0.5,
    label="loc. error theta=[150°,180°]"
)

plt.xlabel('Phi (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error as a function of phi')
plt.grid(True)
plt.legend()
plt.show()
